# Step 2: Immune Density Calculations per Sample
**Purpose:** Take the phenotyped, ROI filtered anndata from step 1 and compute immune cell densities (cells per mm^2) for each ROI, producing a summary table ready for comparison in Step 3

## What this notebook does
1. **Configuration + setup:** Define the sample id, pixel size, input/output paths, and immune populations. The rest of the file paths will be built automatically from these settings
2. **Data loading:** Read the .h5ad file that was saved at the end of Step 1 and make sure the expected columns look correct
3. **ROI area estimation:** For each ROI, fit a convex hull around each cell centroid coords to approximate the tissue area -- then convert from pixels^2 to micrometers^2 to mm^2 
4. **Immune cell counting + density calculations:** — For each immune population within each ROI, count the matching cells by turning the phenotype labels into tokens, and then normalizing by ROI area to produce cells per mm^2 (also compute raw counts and fractions).
5. **Export:** Save the per ROI density table as a csv, with one row per ROI with columns for each immune population's count, fraction, and density.

## Inputs

* `adata_<sample_id>.h5ad` --> the final anndata from Step 1
  
## Outputs

* `<sample_id>_immune_densities.csv` --> the per ROI immune density table with counts, fractions, and cells/mm^2 for each population

## Requirements

* Python 3.10 environment with: scimap, anndata, scanpy, pandas, numpy, scipy, seaborn, matplotlib
* No manual steps required for this part!





In [30]:
# Import libraries

# pandas and numpy: data tables and math
import pandas as pd
import numpy as np

# anndata and scanpy: for the AnnData file format used 
import anndata as ad
import scanpy as sc # for reading/loading it

# scipy: ConvexHull used to estimate ROI area from cell coords
from scipy import stats
from scipy.spatial import ConvexHull

# seaborn and matplotlib mainly for plotting 
import seaborn as sns; sns.set(color_codes=True)
import matplotlib.pyplot as plt

# standard libraries
import re # used later on to split phenotype strings
import os
import sys

# import scimap
import scimap as sm

# Hide common non-critical warnings so our outputs are cleaner
import warnings
warnings.filterwarnings('ignore')

In [31]:
## ‼️UPDATE‼️ this cell when you change samples
# Everything below is the only thing you should need to change between samples

# Sample identifier, used in input filename and output filename
SAMPLE_ID = "BDBAT0085" #update every new sample

# Pixel size from microscope metadata (micrometers/pixel)
PIXEL_SIZE_UM = 0.65   # update if imager settings change

# Path to the .h5ad AnnData file made in Step 1
INPUT_H5AD = f"/Users/elizabethxiu/Downloads/FOR_ELIZABETH/Tutorial/analysis/scimap_results/adata_tocombine/adata_{SAMPLE_ID}.h5ad"
# update this path if drive name or folder structure is changed

# Where to save the output densities CSV
OUTPUT_CSV = f"/Users/elizabethxiu/Downloads/FOR_ELIZABETH_Tutorial/analysis/immune/{SAMPLE_ID}_immune_densities.csv"
# update this path if drive name or folder structure is changed

# Which immune populations to quantify, have to match tokens in the phenotype column of adata.obs. Check!!
IMMUNE_POPS = ['cd8', 'cd4', 'macrophage', 'nk'] # update accordingly

print(f"Configuration loaded for sample: {SAMPLE_ID}")
print(f"Input: {INPUT_H5AD}")
print(f"Output: {OUTPUT_CSV}")
print(f"Populations: {IMMUNE_POPS}")

Configuration loaded for sample: BDBAT0085
Input: /Users/elizabethxiu/Downloads/FOR_ELIZABETH/Tutorial/analysis/scimap_results/adata_tocombine/adata_BDBAT0085.h5ad
Output: /Users/elizabethxiu/Downloads/FOR_ELIZABETH_Tutorial/analysis/immune/BDBAT0085_immune_densities.csv
Populations: ['cd8', 'cd4', 'macrophage', 'nk']


In [32]:
## Load data
# Remember, after step 1, loading adata.obs here will now show spatial coords, phenotype, ROI, and genotype

adata = sc.read_h5ad(INPUT_H5AD) # using the name from the configuration 

# Quick sanity check, confirm that our expected columns exist before running analysis
print(f"Loaded {adata.n_obs:,} cells and {adata.n_vars} markers")
print(f"Columns in adata.obs: {list(adata.obs.columns)}")


Loaded 34,099 cells and 18 markers
Columns in adata.obs: ['X_centroid', 'Y_centroid', 'Area', 'MajorAxisLength', 'MinorAxisLength', 'Eccentricity', 'Solidity', 'Extent', 'Orientation', 'CellID', 'imageid', 'Treatment_Group', 'phenotype', 'ROI1', 'ROI2', 'ROI3', 'ROI4', 'ROI', 'genotype']


In [33]:
## Inspect the metadata
# Remember adata.obs is a pandas data frame; one row per cell and columns = metadata attributes

print(f"Shape: {adata.obs.shape}")
print(f"\nColumn names:{list(adata.obs.columns)}")
print(f"\nPhenotype value counts:")
print(adata.obs['phenotype'].value_counts())
print(f"\nROI count: {adata.obs['ROI'].nunique()} unique ROIs")
print(f"\nGenotypes: {adata.obs['genotype'].unique()}") # returns list of each distinct value

adata.obs.head()  # show the first few rows as table

Shape: (34099, 19)

Column names:['X_centroid', 'Y_centroid', 'Area', 'MajorAxisLength', 'MinorAxisLength', 'Eccentricity', 'Solidity', 'Extent', 'Orientation', 'CellID', 'imageid', 'Treatment_Group', 'phenotype', 'ROI1', 'ROI2', 'ROI3', 'ROI4', 'ROI', 'genotype']

Phenotype value counts:
phenotype
Tumor           26877
CD8+ T cells     3191
CD4+ T cells     2128
Macrophages      1494
NK cells          409
Name: count, dtype: int64

ROI count: 4 unique ROIs

Genotypes: ['test_run']
Categories (1, object): ['test_run']


,X_centroid,Y_centroid,Area,MajorAxisLength,MinorAxisLength,Eccentricity,Solidity,Extent,Orientation,CellID,imageid,Treatment_Group,phenotype,ROI1,ROI2,ROI3,ROI4,ROI,genotype
BDBAT0085_44195,5732.319444,2618.194444,72.0,11.051304,8.734854,0.612602,0.878049,0.600000,-1.153112,44195,BDBAT0085,test_run,CD4+ T cells,Other,ROI2,Other,Other,ROI2,test_run
BDBAT0085_44197,4484.637097,2619.604839,124.0,13.773750,11.461048,0.554635,0.925373,0.738095,-1.149525,44197,BDBAT0085,test_run,Tumor,Other,ROI2,Other,Other,ROI2,test_run
BDBAT0085_44204,4451.633028,2621.596330,109.0,16.005459,9.511900,0.804250,0.832061,0.660606,-0.333483,44204,BDBAT0085,test_run,Tumor,Other,ROI2,Other,Other,ROI2,test_run
BDBAT0085_44205,4458.617284,2621.691358,81.0,14.218614,7.449442,0.851766,0.880435,0.642857,-0.355997,44205,BDBAT0085,test_run,Tumor,Other,ROI2,Other,Other,ROI2,test_run
BDBAT0085_44216,4423.729167,2625.756944,144.0,14.137544,13.123767,0.371852,0.935065,0.734694,-0.425277,44216,BDBAT0085,test_run,Tumor,Other,ROI2,Other,Other,ROI2,test_run


In [34]:
def compute_roi_areas(adata,roi_col='ROI',x_col='X_centroid', y_col='Y_centroid', genotype_col='genotype', pixel_size_um=PIXEL_SIZE_UM): 
    
    """
    Purpose
    -------
    Approximate area of each ROI using a convex hull of cell centroids
    No explicit tissue boundary annotation so the convex hull approximates
    tissue area by wrapping a tight polygon around all detected cells in each ROI

    Parameters
    ----------
    adata : AnnData object from Step 1
    roi_col : column in adata.obs containing ROI labels
    x_col, y_col : column names for cell centroid coords (in pixels)
    genotype_col : column in adata.obs for exp group
    pixel_size_um : micrometers/pixel (set in configuration)

    Returns
    -------
    DataFrame with the columns ROI, Genotype, Area_pixels2, Area_um2, Area_mm2
    """
    obs = adata.obs.copy() # make a copy to work on

    # Force the coordinates to be numeric, they could have been stored as strings
    obs[x_col] = pd.to_numeric(obs[x_col], errors='coerce') # if a value can't be converted to number, just assign NaN (not a number) 
    obs[y_col] = pd.to_numeric(obs[y_col], errors='coerce')

    rows = []

    # split the full dataframe into groups based on column, so each iteration gives a dataframe corresponding to an ROI label 
    for roi, g in obs.groupby(roi_col): 
        pts = g[[x_col, y_col]].dropna().values # extract just the x and y columns and drop any NaN values
                                                # convert to plain numpy array, convex hull needs that

        if pts.shape[0] < 3:
            # we need at least 3 (non collinear) points to define a 2D area
            # ROIs with  less than 3 cells are assigned 0 (skipped, will become NaN in density calc)
            area_px2 = 0.0
            print(f" ROI '{roi}' has only {pts.shape[0]} cells — skipping area calculation")
        else:
            hull = ConvexHull(pts) # fix convex hull around all cell coordinates in this ROI
            area_px2 = float(hull.volume)  # sounds counterintuitive but note that for 2D input,
                                           # scipy uses hull.volume for area and hull.area for perimeter so this is good

        area_um2 = area_px2 * (pixel_size_um ** 2)
        area_mm2 = area_um2 / 1e6 # unit conversions from pixels^2 to micrometers^2 to mm^2

        # Get the most common genotype label in this ROI
        if genotype_col in g.columns:
            gen_vals = g[genotype_col].dropna().astype(str) # drop NaNs, convert to string
            genotype = gen_vals.mode().iloc[0] if not gen_vals.empty else 'unknown' # find most frequently occurring value
                                                                                    # grab the first result (if there's a tie)
                                                                                    # makes sure there are still values left after dropping NaNs
        else:
            genotype = 'unknown'

        rows.append({
            'ROI': roi,
            'Genotype': genotype,
            'Area_pixels2': area_px2,
            'Area_um2': area_um2,
            'Area_mm2': area_mm2 
        }) # add this dictionary of this ROI's results to the "rows" list

    return pd.DataFrame(rows) 

In [35]:
# HELPER FUNCTION
# This takes a cell's phenotype value and splits into list of individual lowercase tokens
# Why? --> could be stored as one string with more than one label, need to split so we can check each
# label individually 
# NOTE: this func isn't meant to be called directly, called automatically inside compute_immune_densities 

def _phen_tokens(x):
    # if the value is missing/NaN, return empty list
    if pd.isna(x):
        return []

    # If somehow the value is already a list/tuple/set just clean + return it
    if isinstance(x, (list, tuple, set)): # this checks if an object belons to a specific class or data type
        return [str(t).strip().lower() for t in x]
    
    s = str(x).strip() # otherwise let's convert the object to string and strip whitespace
    
    if s == '':   # if it's an empty string after stripping, return empty list
        return []
    
    # Now, we split on any of these delimeters (thanks to the re package): ; , | /
    # like "CD8;Ki67" splits like this --> ['CD8', 'Ki67']
    parts = re.split(r'[;,|/]', s) # result is a list
    
    # Strip the whitespace from each token, convert to lowercase, drop empty strings
    # so the list ['CD8', 'Ki67'] becomes ['cd8', 'ki67']
    return [p.strip().lower() for p in parts if p.strip()] # only include each p (each token) if it ISN'T an empty string (can mistakenly happen)

In [36]:
def compute_immune_densities(adata,
                             roi_area_df, 
                             roi_col='ROI',
                             phenotype_col='phenotype',
                             genotype_col='genotype',
                             immune_pops=None):
    """
    Purpose
    -------
    Count the cells of each immune phenotype per ROI and normalize by ROI area; 
    takes the fully phentyped anndata object from the first step and ROI area table from computer_roi_areas
    and makes a summary table of immune infiltration per each ROI

    Parameters
    ----------
    adata : anndata object loaded from step 1
    roi_area_df : output of compute_roi_areas()
    roi_col : column in adata.obs that has the ROI labels
    phenotype_col : column that has cell type labels
    genotype_col  : column that has exp group
    immune_pops   : list of population names to quantify (default to whatever you defined in configuration)

    Returns
    -------
    dataframe with 1 row per ROI and the columns:
    Genotype, ROI, Total_Cells, Area_mm2, cd8_count, cd8_fraction, cd8_cells_per_mm2,
    cd4_count, cd4_fraction, cd4_cells_per_mm2,...etc for each population in immune_pops
    """

    
    # Default to the population list defined in the config cell
    if immune_pops is None:
        immune_pops = IMMUNE_POPS
    immune_pops = [p.lower() for p in immune_pops] # lowercase all pop names so matching isn't case sensitive


    obs = adata.obs.copy() # work on a copy 

    # Here, we must convert the phenotype column from a pandas categorical (done to save memory, turns the column into a lookup table with numbers basically)
    # to a plain object data type with strings -- this forces it to behave like a normal series (1 column, and however many rows) 
    # before applying our phen tokens function
    phen_ser = obs[phenotype_col].astype(object) 
    
    # Apply the _phen_tokens function to every cell's phenotype value
    # This gives a new column '_phen_list' where each entry is a list of those lowercase tokens
    # Remember the example from before: "CD8;Ki67" becomes ['cd8', 'ki67']
    obs['_phen_list'] = phen_ser.apply(_phen_tokens)

    # This list will collect 1 dictionary per ROI
    densities_rows = []

    # Again, split full cell table into groups by ROI
    # For each iteration, roi = ROI label and g = the dataframe of only that ROI's cells
    for roi, g in obs.groupby(roi_col):
        
        total_cells = len(g) # total num of cells in this ROI

        genotype = 'unknown' # get the genotype for this ROI (the most common value)
        if genotype_col in g.columns:
            gen_vals = g[genotype_col].dropna().astype(str)
            if not gen_vals.empty:
                genotype = gen_vals.mode().iloc[0] # again, grab the first value if there's a tie and make sure there are actually values to grab

        # Look up this ROI's area from the roi_area_df 
        arow = roi_area_df[roi_area_df['ROI'] == roi] # get which row matchtes the current ROI
                                                      #another use of a boolean mask, keep the rows where it's True
        if arow.empty:
            # the ROI wasn't found in the area table so density will be NaN
            area_mm2 = np.nan
        else:
            area_mm2 = float(arow['Area_mm2'].values[0]) # grab the 'area_mm2' column from this row, 
                                                         # convert to a numpy array and get the first (only) element 
            if area_mm2 == 0:
                # 0 area would produce infinite density!! Use NaN instead
                area_mm2 = np.nan

        row = { # start making this ROI's result dictionary
            'Genotype': genotype,
            'ROI': roi,
            'Total_Cells': total_cells,
            'Area_mm2': area_mm2
        }

        # Nested loop here - now within this ROI iteratoin, loop through each immune population and compute the 3 metrics we need
        for pop in immune_pops:
            
            # Count the cells where this pop name appears in the token list
            # lambda checks -- does 'cd8' exactly equal any token OR appear inside any token?
            # Example --> if the pop='cd8' and the tokens = ['cd8','ki67'], then'cd8' == 'cd8' --> True, counted!
            # Or, if the pop='cd8', the tokens =['cd8_tcf7'], then 'cd8' is in 'cd8_tcf7' --> True, counted as well!
            
            count = int(
                g['_phen_list'].apply(
                    lambda tokens: any(pop == t or pop in t for t in tokens) # returns True if at least one of these things is true
                ).sum()  # this counts the number of True values
            )
            
            # Fraction of all cells in this ROI that are this pop
            frac = count / total_cells if total_cells > 0 else np.nan
            
            # Density is cells per mm^2 
            density = count / area_mm2 if (area_mm2 is not None and not np.isnan(area_mm2)) else np.nan # make sure area isn't None or NaN

            # Add all three metrics to the row dictionary
            row[f'{pop}_count'] = count # how many cells in this immune population
            row[f'{pop}_fraction'] = frac   # what fraction they make up out of the total cells
            row[f'{pop}_cells_per_mm2'] = density # density of this immune population in this ROI

        densities_rows.append(row)

    # Convert the list of dictionaries into a dataframe, 1 row per ROI
    densities_df = pd.DataFrame(densities_rows)
    return densities_df

In [37]:
## Run analyses

print("Computing ROI areas...")
roi_area_df = compute_roi_areas(adata, pixel_size_um=PIXEL_SIZE_UM) # call compute_roi_areas() functin on the loaded adata object

# Print the full area table, it should be prettyshort (1 row per ROI)
# Check that Area_mm2 values look reasonable (not 0) 
print(f"\nROI areas ({len(roi_area_df)} ROIs):")
print(roi_area_df.to_string(index=False)) # show as plain text, hide indices (row numbers on the side)


print("\nComputing immune cell densities...")
densities_df = compute_immune_densities( # call the compute_immune_densities() function using the adata cull dataset and roi_area_df that we just computed above
    adata,
    roi_area_df,
    roi_col='ROI',
    phenotype_col='phenotype',
    genotype_col='genotype',
    immune_pops=IMMUNE_POPS # the list defined in configuration cell
)

# Preview the first few rows of the output table
# Again, the columns will be: ROI, Genotype, Total_Cells, Area_mm2,cd8_count, cd8_fraction, cd8_cells_per_mm2, etc for each pop
print(f"\nDensities computed for {len(densities_df)} ROIs.")
densities_df.head()

Computing ROI areas...

ROI areas (4 ROIs):
 ROI Genotype  Area_pixels2     Area_um2  Area_mm2
ROI1 test_run  1.308469e+06 5.528280e+05  0.552828
ROI2 test_run  1.420490e+06 6.001569e+05  0.600157
ROI3 test_run  4.467545e+06 1.887538e+06  1.887538
ROI4 test_run  2.456402e+06 1.037830e+06  1.037830

Computing immune cell densities...

Densities computed for 4 ROIs.


,Genotype,ROI,Total_Cells,Area_mm2,cd8_count,cd8_fraction,cd8_cells_per_mm2,cd4_count,cd4_fraction,cd4_cells_per_mm2,macrophage_count,macrophage_fraction,macrophage_cells_per_mm2,nk_count,nk_fraction,nk_cells_per_mm2
0,test_run,ROI1,6764,0.552828,810,0.119752,1465.193596,677,0.100089,1224.612426,506,0.074808,915.293777,140,0.020698,253.243338
1,test_run,ROI2,5372,0.600157,380,0.070737,633.167765,145,0.026992,241.603489,376,0.069993,626.502841,64,0.011914,106.638782
2,test_run,ROI3,18072,1.887538,1929,0.106740,1021.966378,754,0.041722,399.462234,483,0.026726,255.888938,148,0.008189,78.409033
3,test_run,ROI4,3891,1.037830,72,0.018504,69.375531,552,0.141866,531.879072,129,0.033153,124.297827,57,0.014649,54.922295


In [38]:
## Save the immune density table output as a csv -- will be the handoff to step 3

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True) # create output folder for the csv if doesn't already exist

# Save to csv, index=False means don't write the row numbers as a whole separate column
densities_df.to_csv(OUTPUT_CSV, index=False)

# Confirmation summary to make sure everything went well!
print(f"Saved: {OUTPUT_CSV}")
print(f"Rows (ROIs): {densities_df.shape[0]}") # the first index returns the number of rows, or ROIs here
print(f"Columns: {densities_df.shape[1]}") # second index --> number of columns
print(f"\nColumn list:")
print(list(densities_df.columns))
print("\nStep 2 complete!")
print(f"Sample: {SAMPLE_ID}")
print(f"Next: open the step 3 notebook and load this CSV!")

Saved: /Users/elizabethxiu/Downloads/FOR_ELIZABETH_Tutorial/analysis/immune/BDBAT0085_immune_densities.csv
Rows (ROIs): 4
Columns: 16

Column list:
['Genotype', 'ROI', 'Total_Cells', 'Area_mm2', 'cd8_count', 'cd8_fraction', 'cd8_cells_per_mm2', 'cd4_count', 'cd4_fraction', 'cd4_cells_per_mm2', 'macrophage_count', 'macrophage_fraction', 'macrophage_cells_per_mm2', 'nk_count', 'nk_fraction', 'nk_cells_per_mm2']

Step 2 complete!
Sample: BDBAT0085
Next: open the step 3 notebook and load this CSV!
